# 4. Method anchor

Part of the **[Fig. 1 chapter](fig1.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

import warnings
warnings.filterwarnings("ignore")


In [ ]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [ ]:
indir = f'{ENTEX_ROOT}/'


In [ ]:
L1_meta = pd.read_csv(f'{indir}L1color.tsv', sep='\t', header=0, index_col=0)
L1_meta = L1_meta.drop(['c35', 'c36'], axis=0)
L1_annot = L1_meta['L1_abbr'].to_dict()
L1_color = L1_meta['color'].to_dict()


In [ ]:
meta = pd.read_csv(f'{indir}clustering/merged/5kCG100k3C_summary.csv.gz', header=0, index_col=0)
meta
# mcad = anndata.AnnData(obs=mcad.obs, obsm=mcad.obsm)
# mcad.write_h5ad('cell_86689_16tissue_100k3C_autosomal.h5ad')

In [ ]:
ds = 1
coord_base = 'tsne'
for g in ['EG', 'TrCo']:
    mcad = anndata.read_h5ad(f'{indir}clustering/tissue/L1/{g}/mCG_5kb_lsi.h5ad')
    print(g)
    npc = significant_pc_test(mcad, p_cutoff=0.05, obsm='lsi_all', update=False)
    # if f'5kCG_u{npc}_seuratcc{npc}_tsne' not in mcad.obsm.keys():
    #     mcad.obsm[f'5kCG_u{npc}_seuratcc{npc}_tsne'] = mcad.obsm[f'lsi_u{npc}_seurat_cc{npc}_tsne'].copy()
    #     mcad.obsm[f'5kCG_u{npc}_seuratcc{npc}'] = mcad.obsm[f'lsi_seurat_cc{npc}'].copy()
    #     del mcad.obsm[f'lsi_u{npc}_seurat_cc{npc}_tsne'], mcad.obsm[f'lsi_seurat_cc{npc}']
    #     mcad.write_h5ad(f'{indir}clustering/tissue/{g}/mCG_5kb_lsi.h5ad')

    fig, axes = plt.subplots(2, 3, figsize=(12, 6), dpi=300, constrained_layout=True)
    fig.suptitle(g, fontsize=20)
    
    for i,method in enumerate([f'_seuratcc{npc}', f'_seuratcc{npc}filter60', f'_seuratcc{npc}geofilter60']):
        
        mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'5kCG_u{npc}{method}_tsne'].copy()
        dump_embedding(mcad, coord_base)
        mcad.obs['celltype'] = meta.loc[mcad.obs.index, 'majortype'].map(L1_annot)
        count = mcad.obs['celltype'].value_counts()
        selct = count.index[count>=20]
        tmp = mcad.obs.loc[mcad.obs['celltype'].isin(selct)]
        ax = axes[0,i]
        _ = categorical_scatter(data=tmp,
                                ax=ax,
                                coord_base=coord_base,
                                hue='Donor',
                                s=ds,
                                labelsize=8,
                                max_points=None,
                                palette='tab10',
                                scatter_kws={'rasterized':True},
                                show_legend=(i==2))
        ax.set_title(['ALLCools', 'ALLCools Dist60', 'ALLCools Geodist60'][i])

        ax = axes[1,i]
        _ = categorical_scatter(data=tmp,
                                ax=ax,
                                coord_base=coord_base,
                                hue='celltype',
                                # text_anno='celltype', 
                                s=ds,
                                labelsize=8,
                                max_points=None,
                                palette='tab20',
                                scatter_kws={'rasterized':True},
                                legend_kws={'ncol':1}, 
                                show_legend=(i==2))
        
    fig.savefig(f'clustering_summary/mc_lsi_anchormethod_{g}.pdf', transparent=True)
        
        

In [ ]:
knn = 25
import pynndescent
from sklearn.metrics import pairwise_distances
from scipy.sparse import csr_matrix

In [ ]:
def bandwidth(pc):
    ncell = pc.shape[0]
    print('Dist')
    D = pairwise_distances(pc, metric='euclidean')
    print('Sort')
    #G1 = np.argsort(D, axis=1)[:, 1:knn+1]
    index = pynndescent.NNDescent(pc, metric='euclidean', n_neighbors=knn+1)
    G = index.neighbor_graph[0][:, 1:]
    d = np.array([D[i,G[i,0]] for i in range(ncell)])
    G = csr_matrix((np.ones(knn*ncell), G.flatten(), np.array([knn*i for i in range(ncell+1)])), shape=(ncell, ncell))
    print('Jacc')
    inter = G.dot(G.T)
    diag = inter.diagonal()
    jac = csr_matrix(inter.astype(float) / (diag[None,:] + diag[:,None] - inter))
    jac.data = 1 - jac.data
    print('Sort')
    #idx = np.argsort(jac.toarray(), axis=1)[:, -20:]
    #bw1 = np.mean([D[i,idx[i]] for i in range(ncell)], axis=1)
    jac = jac.toarray()
    idx = np.argsort(jac, axis=1)[:, -21:]
    bw = np.zeros(ncell)
    for i in range(ncell):
        if jac[i, idx[i,0]]==jac[i, idx[i,1]]:
            tmp = np.sort(D[i, jac[i]>=jac[i, idx[i, 0]]])
            bw[i] = np.mean(tmp[-20:])
        else:
            bw[i] = np.mean(D[i,idx[i, 1:]])
    return d,bw,G




In [ ]:
def wnn(pc_list):
    nm = len(pc_list)
    nc = pc_list[0].shape[0]
    pc_list = [normalize(pc, axis=1) for pc in pc_list]
    d, bw, G = [], [], []
    for i,pc in enumerate(pc_list):
        tmp = bandwidth(pc)
        d.append(tmp[0])
        bw.append(tmp[1])
        G.append(tmp[2])
        print(i)
    w = np.zeros((nm, nc))
    D = np.zeros((nc, nc))
    for i in range(nm):
        tmp = G[i].dot(pc_list[i]) / knn
        t11 = np.sqrt(np.square(pc_list[i]-tmp).sum(axis=1))
        t11 = np.exp(np.where((t11 - d[i]) > 0, t11 - d[i], 0) / (d[i] - bw[i]))
        for j in range(nm):
            if i!=j:
                tmp = G[j].dot(pc_list[i]) / knn
                t12 = np.sqrt(np.square(pc_list[i]-tmp).sum(axis=1))
                t12 = np.exp(np.where((t12 - d[i]) > 0, t12 - d[i], 0) / (d[i] - bw[i]))
                s = np.exp(t11 / (t12 + 1e-4))
                s[s>200] = 200
                w[i] += s
        print('Dist')
        tmp = pairwise_distances(pc_list[i], metric='euclidean')
        print('Weight')
        tmp = tmp - d[i]
        D += np.exp(np.where(tmp > 0, tmp, 0) / (d[i] - bw[i])) * w[i]
        print(i)
    D = D / np.sum(w, axis=0)
    w = w / np.sum(w, axis=0)
    return w, D.T


In [ ]:
indir = f'{ENTEX_ROOT}/clustering/tissue/L1/TrCo/'
outdir = f'{ENTEX_ROOT}/analysis/method/'

In [ ]:
adata_mc = anndata.read_h5ad(f'{indir}mCG_5kb_lsi.h5ad')
adata_3c = anndata.read_h5ad(f'{indir}HiC_100kb_pca.h5ad')
adata_3c = adata_3c[adata_mc.obs.index].copy()


In [ ]:
pc_list = [adata_mc.obsm['5kCG_u21_seuratL2'].copy(), 
           adata_3c.obsm['100k3C_pc14_seuratL2'].copy(), 
          ]
w, D = wnn(pc_list)



In [ ]:
adata = anndata.read_h5ad(f'{indir}5kCG100k3C_embed.h5ad')
adata

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=50, svd_solver='arpack')
adata.obsm['wnn_pc_all'] = pca.fit_transform(D.T)


In [ ]:
npc = significant_pc_test(adata, p_cutoff=0.05, update=False, obsm='wnn_pc_all')


In [ ]:
adata.obsm['X_pca'] = normalize(adata.obsm['wnn_pc_all'][:, :npc], axis=1)


In [ ]:
tsne(adata, obsm='X_pca', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
adata.obsm[f'wnn_pc{npc}_tsne'] = adata.obsm['X_tsne'].copy()


In [ ]:
res = 1.0
for dimred,tmp in zip(['5kCG_u21_seuratL2', '100k3C_pc14_seuratL2'], 
                      [adata_mc, adata_3c]):
    sc.pp.neighbors(tmp, n_neighbors=knn, use_rep=dimred)
    sc.tl.leiden(tmp, resolution=res, flavor='igraph')
    tmp.obs[f'{dimred}_leiden{res}'] = tmp.obs['leiden'].copy()
    

In [ ]:
adata.obs['leiden_mc'] = adata_mc.obs['leiden'].copy()
adata.obs['leiden_3c'] = adata_3c.obs['leiden'].copy()


In [ ]:
adata

In [ ]:
ds = 4
coord_base = 'tsne'

fig, axes = plt.subplots(2, 2, figsize=(10, 8), dpi=300, constrained_layout=True)
for i,dim_coord in enumerate(['wnn_pc45', '5kCG100k3C_u21pc14']):
    for j,dim_leiden in enumerate(['leiden_mc', 'leiden_3c']):
        ax = axes[i,j]
        adata.obsm[f'X_{coord_base}'] = adata.obsm[f'{dim_coord}_{coord_base}']
        adata = dump_embedding(adata, coord_base)
        ax.set_title(f'{dim_coord} {dim_leiden}')
        _ = categorical_scatter(data=adata.obs,
                                ax=ax,
                                coord_base=coord_base,
                                hue=dim_leiden,
                                s=ds,
                                labelsize=8,
                                max_points=None,
                                palette='tab20',
                                scatter_kws={'rasterized':True},
                               )

fig.savefig(f'{outdir}TrCo_wnn_concat_leiden.pdf', transparent=True)


In [ ]:
mcad.obsm['X_tsne'] = mcad.obsm['wnn_pc15_tsne'].copy()
dump_embedding(mcad, 'tsne')

coord_base = 'tsne'
fig, axes = plt.subplots(2, 3, figsize=(18, 8), dpi=300, constrained_layout=True)
_ = categorical_scatter(data=mcad.obs,
                        ax=axes[0,0],
                        coord_base=coord_base,
                        hue='Sample',
                        labelsize=10,
                        show_legend=True)
_ = continuous_scatter(ax=axes[0,1],
                       data=mcad.obs,
                       hue='FinalmCReads',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=10,
                       s=4)
'''
_ = categorical_scatter(data=mcad.obs,
                        ax=axes[0,2],
                        coord_base='umap',
                        hue='Leiden',
                        text_anno=f'Leiden',
                        labelsize=10,
                        show_legend=True)
                        '''
_ = categorical_scatter(data=mcad.obs,
                        ax=axes[0,2],
                        coord_base=coord_base,
                        hue='Cluster',
                        text_anno='Cluster',
                        labelsize=10,
                        show_legend=True)

_ = continuous_scatter(ax=axes[1,0],
                       data=mcad.obs,
                       hue='Cis/Trans',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=10,
                       s=4)
_ = continuous_scatter(ax=axes[1,1],
                       data=mcad.obs,
                       hue='mCHFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=10,
                       s=4)
_ = continuous_scatter(ax=axes[1,2],
                       data=mcad.obs,
                       hue='mCGFrac',
                       coord_base=coord_base,
                       max_points=None,
                       labelsize=10,
                       s=4)



In [ ]:
mcad.obsm['5kCG100k3C_pc'] = np.hstack([mcad.obsm['5kCG_u15hm'] / mcad.obsm['5kCG_u15hm'].std(), 
                                        mcad.obsm['100k3C_u15hm'] / mcad.obsm['100k3C_u15hm'].std()])
tsne(mcad, obsm='5kCG100k3C_pc', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm['5kCG100k3C_pc_tsne'] = mcad.obsm['X_tsne'].copy()


In [ ]:
coord_base = 'tsne'
fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=300, constrained_layout=True)
for i,coord in enumerate(['wnn_pc15', '5kCG100k3C_pc', '5kCG_u15hm', '100k3C_u15hm']):
    mcad.obsm['X_tsne'] = mcad.obsm[f'{coord}_tsne'].copy()
    dump_embedding(mcad, 'tsne')
    ax = axes.flatten()[i]
    ax.set_title(coord.split('_')[0].upper(), fontsize=10)
    _ = categorical_scatter(data=mcad.obs,
                            ax=ax,
                            coord_base=coord_base,
                            hue='Cluster',
                            text_anno='Cluster',
                            labelsize=10,
                            palette='tab20',
                            show_legend=(i==3)
                           )




In [ ]:
from scanpy.neighbors.__init__ import _get_sparse_matrix_from_indices_distances_umap, _compute_connectivities_umap

sample_range = np.arange(D.shape[0])[:, None]
indices = np.argpartition(-D, knn+1, axis=1)[:, :knn+1]
indices = indices[sample_range, np.argsort(-D[sample_range, indices])]
distances = -D[sample_range, indices[:, 1:]]
indices = indices[:,1:]
distances, connectivities = _compute_connectivities_umap(indices, distances, D.shape[0], knn)

mcad.obsp['connectivities'] = connectivities.copy()
mcad.obsp['distances'] = distances.copy()

sc.tl.umap(mcad, random_state=0)
dump_embedding(mcad, 'umap')
mcad.obsm['wnn_umap'] = mcad.obsm['X_umap'].copy()


In [ ]:
res = 0.5
sc.tl.leiden(mcad, resolution=res)
mcad.obs[f'wnn_leiden_{res}'] = mcad.obs['leiden'].copy()
mcad.obs[f'wnn_leiden_{res}']


In [ ]:
knn = 25
res = 0.5
for coord in ['wnn_pc15', '5kCG100k3C_pc', '5kCG_u15hm', '100k3C_u15hm']:
    sc.pp.neighbors(mcad, n_neighbors=knn, use_rep=coord)
    sc.tl.leiden(mcad, resolution=res)
    mcad.obs[f'{coord}_leiden_{res}'] = mcad.obs['leiden'].copy()


In [ ]:
coord_base = 'tsne'
fig, axes = plt.subplots(4, 4, figsize=(12, 8), dpi=300, constrained_layout=True)
for i,coord1 in enumerate(['wnn_pc15', '5kCG100k3C_pc', '5kCG_u15hm', '100k3C_u15hm']):
    mcad.obsm['X_tsne'] = mcad.obsm[f'{coord1}_tsne'].copy()
    dump_embedding(mcad, 'tsne')
    for j,coord2 in enumerate(['wnn', '5kCG100k3C_pc', '5kCG_u15hm', '100k3C_u15hm']):
        ax = axes[j,i]
        if j==0:
            ax.set_title(coord1.split('_')[0].upper(), fontsize=10)
        _ = categorical_scatter(data=mcad.obs,
                                ax=ax,
                                coord_base=coord_base,
                                hue=f'{coord2}_leiden_{res}',
                                text_anno=f'{coord2}_leiden_{res}',
                                labelsize=10,
                                palette='tab20',
                               )


     
